In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from torchvision.datasets import CIFAR10
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
trainset = CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = CIFAR10(root='./data', train=False, download=True, transform=transform)

Mounted at /content/drive


100%|██████████| 170M/170M [01:01<00:00, 2.79MB/s]


In [ ]:
from torch.utils.data import DataLoader
trainloader = DataLoader(trainset,batch_size=64,shuffle = True)
testloader = DataLoader(testset,batch_size=64)

In [ ]:
class CNN(nn.Module):
  def __init__(self):
    super(CNN, self).__init__()

    self.conv_layers = nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2, 2),

        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2, 2),

        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2, 2),
    )

    self.fc_layers = nn.Sequential(
        nn.Linear(4*4*128, 256),
        nn.ReLU(),
        nn.Linear(256, 10)
    )

  def forward(self, x):
    x = self.conv_layers(x)
    x = x.view(x.size(0), -1)
    x = self.fc_layers(x)
    return x

In [ ]:
model = CNN()

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
epochs = 1
for epoch in range(epochs):
  epoch_training_loss = 0.0

  for images, labels in trainloader:
    optimizer.zero_grad()

    output = model.forward(images)
    loss = criterion(output,labels)
    loss.backward()
    optimizer.step()

    epoch_training_loss += loss.item()

  print(f"epoch = {epoch} & loss {epoch_training_loss/len(trainloader)}")

epoch = 0 & loss 0.3506356428003372


In [ ]:
corrected_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
  for images , labels in testloader:
    outputs = model.forward(images)
    _,predicted = torch.max(outputs,1)

    corrected_labels += (predicted == labels).sum().item()
    total_labels += labels.size(0)


print(f"Accuracy : {corrected_labels/total_labels * 100}")

Accuracy : 74.97
